In [1]:
import os
from google.colab import userdata
from openai import OpenAI

# Option 1: Set as an environment variable (recommended for OpenAI SDK, LangChain, etc.)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()




In [2]:
!pip install -qU langchain langchain-core langchain-openai tiktoken

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-6.1-terra",  # Or "gpt-6.1-sol" for complex reasoning

)
response = llm.invoke("Explain Recursion in one sentence")

# Print the text reply
print(response.content)

Recursion is when a function solves a problem by calling itself on smaller versions of that problem until it reaches a stopping condition.


In [9]:
# 2. Initialize ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
llm = ChatOpenAI(model="gpt-4o-mini")

# 3. Build a conversation using explicit message classes
messages = [
    SystemMessage(content="You are a concise Python tutor."),
    HumanMessage(content="What is a list comprehension?"),
    AIMessage(content="It is a compact one-line syntax for creating a new list from an existing iterable."),
    HumanMessage(content="Show me a quick example that squares numbers 1 to 5.")
]

# 4. Invoke the model (returns an AIMessage object)
response = llm.invoke(messages)

print(response.content)

Certainly! Here’s an example of a list comprehension that squares the numbers from 1 to 5:

```python
squares = [x**2 for x in range(1, 6)]
print(squares)
```

This will output:

```
[1, 4, 9, 16, 25]
```


In [10]:
messages.append(response)
messages.append(HumanMessage(content="Now filter that list to only keep even squares."))

next_response = llm.invoke(messages)
print(next_response.content)

You can modify the list comprehension to include a condition that filters for even squares. Here's how it looks:

```python
even_squares = [x**2 for x in range(1, 6) if x**2 % 2 == 0]
print(even_squares)
```

This will output:

```
[4, 16]
```


In [11]:
from langchain_core.prompts import ChatPromptTemplate



# 2. Initialize the model
llm = ChatOpenAI(model="gpt-4o-mini")

# 3. Create a reusable prompt template with {variable} placeholders
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful translator. Translate the user's text into {language}."),
    ("human", "{text}")
])

# 4. Chain the prompt and the model together using LCEL (|)
chain = prompt | llm

# 5. Run the chain by passing a dictionary for the placeholders
response = chain.invoke({
    "language": "Tamil",
    "text": "Good morning! How are you doing today?"
})

print(response.content)

காலை வணக்கம்! நீங்கள் இன்று எப்படி இருக்கிறீர்கள்?


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("Give me a one-sentence fun fact about {topic}.")

# Chain prompt -> model -> string parser
chain = prompt | llm | StrOutputParser()

# Returns a plain Python string directly (no .content needed)
result = chain.invoke({"topic": "space"})
print(result)

A teaspoon of a neutron star would weigh about 6 billion tons, making it one of the densest substances in the universe!


In [5]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Define your desired output structure
class PlayerDetails(BaseModel):
    Name: str = Field(description="The name of the player")
    DOB: str = Field(description="Date Of Birth")


# 2. Bind the schema to ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")
structured_llm = llm.with_structured_output(PlayerDetails)

# 3. Chain with a prompt and invoke
prompt = ChatPromptTemplate.from_template("Extract Player details from this text:\n{review_text}")
chain = prompt | structured_llm

result = chain.invoke({
    "review_text": "Sachin Tendulkar was born on 24-April-1973."
})

# Access fields directly as Python attributes or convert to a dict/JSON
print(result.Name)        # Inception
print(result.DOB)       # 9
print(result.model_dump()) # {'title': 'Inception', 'sentiment': 'positive', 'rating': 9, 'summary': '...'}


Sachin Tendulkar
24-April-1973
{'Name': 'Sachin Tendulkar', 'DOB': '24-April-1973'}


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel


llm = ChatOpenAI(model="gpt-4o-mini")

# 1. Basic sequential Runnable chain using the pipe (|) operator
fact_prompt = ChatPromptTemplate.from_template("Give me a one-sentence fun fact about {topic}.")
fact_chain = fact_prompt | llm | StrOutputParser()

# 2. Wrap a regular Python function as a Runnable using RunnableLambda
format_output = RunnableLambda(lambda text: f"DID YOU KNOW?\n{text.upper()}")

# 3. Combine the fact chain with the custom Python Runnable
styled_fact_chain = fact_chain | format_output

print(styled_fact_chain.invoke({"topic": "BING.COM"}))


DID YOU KNOW?
BING.COM WAS LAUNCHED BY MICROSOFT IN 2009 AND ORIGINALLY AIMED TO DIFFERENTIATE ITSELF WITH FEATURES LIKE THE "BING REWARDS" PROGRAM, WHICH ALLOWED USERS TO EARN POINTS FOR SEARCHING THAT COULD BE REDEEMED FOR GIFT CARDS AND OTHER PRIZES.


In [23]:
# Prompt to translate the generated fact into Tamil
translate_prompt = ChatPromptTemplate.from_template("Translate this into Tamil:\n{fact}")
translate_chain = translate_prompt | llm | StrOutputParser()

# Run fact_chain first, then pass its output into two parallel branches
full_pipeline = (
    fact_chain
    | RunnableParallel(
        original_fact=RunnablePassthrough(),               # Keeps the English fact as-is
        tamil_translation={"fact": RunnablePassthrough()} | translate_chain  # Pipes it into the translator
    )
)

result = full_pipeline.invoke({"topic": "Siruseri"})

print("English:", result["original_fact"])
print("Tamil:", result["tamil_translation"])


English: Siruseri is home to one of India’s largest IT parks, the SIPCOT IT Park, which hosts over 50 leading tech companies and plays a vital role in the country’s technology landscape.
Tamil: சிறுசேரி இந்தியாவின் மிகப்பெரிய ஐடி பூங்காக்களில் ஒன்றான SIPCOT ஐடி பூங்காவின் இல்லமாகும், இது 50 ஆயிற்று முன்னணி தொழில்நுட்ப நிறுவனங்களைப் பொருத்தமாக்குகிறது மற்றும் நாட்டின் தொழில்நுட்ப காட்சியில் முக்கிய பங்கு வகிக்கிறது.


In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def multiply(a: int, b: int) -> str:
    """Division."""
    try:
        return a / b
    except ZeroDivisionError:
        return "Cannot divide by zero"
    return a / b

# 3. Initialize ChatOpenAI and bind the tool
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools([multiply])

# 4. Ask a question that requires the tool
messages = [HumanMessage(content="What is 4 divided by 2?")]
ai_msg = llm_with_tools.invoke(messages)

# Append the model's tool-call request to the message history
messages.append(ai_msg)

# 5. Check if the model requested a tool call and execute it
if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        print("Model requested tool:", tool_call["name"], "with args:", tool_call["args"])

        # Run the actual Python function
        tool_result = "Cannot divide by zero"#multiply.invoke(tool_call["args"])
        print(tool_result)
        # Wrap the output in a ToolMessage using the matching tool_call_id
        messages.append(
            ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
        )

    # 6. Send the tool result back to the model for the final natural-language answer
    final_response = llm_with_tools.invoke(messages)
    print(final_response)
    print("\nFinal Answer:", final_response.content)

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage



# 2. Define the division tool with built-in ZeroDivisionError handling
@tool
def divide(a: float, b: float) -> str:
    """Divide the first number (a) by the second number (b)."""
    try:
        result = a / b
        return str(result)
    except ZeroDivisionError:
        return "Error: Division by zero is undefined. Please provide a non-zero divisor."

# 3. Initialize ChatOpenAI and bind the tool
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools([divide])

def ask_calculator(question: str):
    messages = [HumanMessage(content=question)]
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)

    # 4. Execute any requested tool calls
    if ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            try:
                tool_output = divide.invoke(tool_call["args"])
                is_error = str(tool_output).startswith("Error:")

                messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        status="error" if is_error else "success"
                    )
                )
            except Exception as e:
                # Fallback catch for unexpected runtime errors
                messages.append(
                    ToolMessage(
                        content=f"Tool execution failed: {str(e)}",
                        tool_call_id=tool_call["id"],
                        status="error"
                    )
                )

        # 5. Send the tool outcome back to the model for a user-friendly response
        final_response = llm_with_tools.invoke(messages)
        return final_response.content

    return ai_msg.content

# Test 1: Normal division
print("Test 1:", ask_calculator("Explain in one sentence. Are you using any tool if yes with name?"))

# Test 2: Divide by zero
print("Test 2:", ask_calculator("What is 50 divided by 0?"))
# Test 3: No tools
print("Test 1:", ask_calculator("Explain in one sentence. Are you using any tool if yes with name?"))



Test 1: Yes, I am using the function tool to perform mathematical operations, specifically the "divide" function.
Test 2: Dividing by zero is undefined in mathematics. Therefore, 50 divided by 0 does not yield a valid result.
Test 1: Yes, I'm using a tool called "functions.divide" to perform division operations.


In [36]:

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


# 2. Create a prompt with a placeholder for past chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, concise assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | llm

# 3. Create a dictionary store and a lookup function for session histories
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 4. Wrap the chain with RunnableWithMessageHistory
chat_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",      # Matches {input} in the prompt
    history_messages_key="history"   # Matches variable_name="history" in MessagesPlaceholder
)

# 5. Configure a session_id so memory stays isolated per user/chat
config = {"configurable": {"session_id": "user_101"}}

# Turn 1: Introduce information
res1 = chat_with_memory.invoke({"input": "Hi! I have Permanent Id card but I forgot to bring.I received a temporary ID card only for today from Admin."}, config=config)
print("AI recording:", res1.content)

# Turn 2: Ask a follow-up (the model remembers Turn 1 automatically)
res2 = chat_with_memory.invoke({"input": "Do I have an identity card?"}, config=config)
print("AI recalling:", res2.content)

# Turn 3: Switch to a different session_id (starts fresh with empty memory)
new_config = {"configurable": {"session_id": "user_202"}}
res3 = chat_with_memory.invoke({"input": "Did I bring Id Card?"}, config=new_config)
print("AI (New Session):", res3.content)


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AI recording: If you have a temporary ID card from Admin, you should be able to use that for the day. Make sure to check if there are any specific guidelines for using temporary IDs in your situation. If you're facing any issues, it may be helpful to explain your circumstance to the relevant authorities.
AI recalling: Based on your earlier message, you have a Permanent ID card, but you forgot to bring it today. You also received a temporary ID card for today from Admin. So yes, you have an ID card, but it's just the temporary one for today.
AI (New Session): I don't have access to your personal items, so I can't tell if you brought your ID card. You might want to check your bag or pockets.


In [38]:

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.runnables import RunnableParallel, RunnableLambda



# 2. Define simple helper tools for the agents to use
@tool
def check_character_count(text: str) -> str:
    """Count the exact number of characters in a draft post."""
    count = len(text)
    status = "Within 280 limit" if count <= 280 else "Exceeds 280 limit!"
    return f"Character count: {count} ({status})"

@tool
def get_trending_hashtags(platform: str) -> str:
    """Return popular engagement hashtag styles for a given social platform."""
    styles = {
        "twitter": "#TechTrends #AI #BuildInPublic",
        "instagram": "#Inspiration #DailyVibes #InstaGood #BehindTheScenes",
        "facebook": "#Community #Discussion #LearnTogether"
    }
    return styles.get(platform.lower(), "#Trending #SocialMedia")

# 3. Initialize the shared OpenAI model
llm = ChatOpenAI(model="gpt-4o-mini")

# 4. Create the 3 specialized agents using create_agent()
twitter_agent = create_agent(
    model=llm,
    tools=[check_character_count, get_trending_hashtags],
    system_prompt=(
        "You are a Twitter/X content specialist. Write a punchy, engaging tweet on the user's topic. "
        "Always check your draft with check_character_count to guarantee it is under 280 characters. "
        "Return only the final tweet."
    )
)

instagram_agent = create_agent(
    model=llm,
    tools=[get_trending_hashtags],
    system_prompt=(
        "You are an Instagram creator. Write a visually descriptive caption with emojis, a strong hook, "
        "a call-to-action to comment, and 4-5 relevant hashtags. Return only the final caption."
    )
)

facebook_agent = create_agent(
    model=llm,
    tools=[get_trending_hashtags],
    system_prompt=(
        "You are a Facebook community manager. Write a warm, conversational, story-style post "
        "that encourages friends and followers to share their thoughts in the comments. Return only the final post."
    )
)

# 5. Helper wrapper to format the user topic into agent messages and extract the final text
def run_agent(agent, topic: str) -> str:
    result = agent.invoke({
        "messages": [{"role": "user", "content": f"Create a social media post about: {topic}"}]
    })
    return result["messages"][-1].content

# 6. Run all 3 agents in parallel on the same topic
social_media_team = RunnableParallel(
    twitter=RunnableLambda(lambda topic: run_agent(twitter_agent, topic)),
    instagram=RunnableLambda(lambda topic: run_agent(instagram_agent, topic)),
    facebook=RunnableLambda(lambda topic: run_agent(facebook_agent, topic))
)

# Test with a user topic
user_topic = "Tony Fergusson Vs Khabib Nurmagamedov Dream Match that never happened"
posts = social_media_team.invoke(user_topic)

print("=== TWITTER / X ===")
print(posts["twitter"], "\n")

print("=== INSTAGRAM ===")
print(posts["instagram"], "\n")

print("=== FACEBOOK ===")
print(posts["facebook"])


=== TWITTER / X ===
💥 Fight fans, imagine the clash of titans: Tony Ferguson vs. Khabib Nurmagomedov! 😱 This dream match never happened, but it’s forever etched in our minds. Would Tony’s unorthodox style have triumphed over Khabib's grappling genius? 🤔 What do you think? 🔥 #UFC #MMA #DreamMatch 

=== INSTAGRAM ===
🔥 A dream fight that never materialized! Imagine the high-octane showdown between Tony Ferguson and Khabib Nurmagomedov – two legends, one octagon! 🥊✨ The chaos of Tony's unorthodox style against Khabib's relentless grappling prowess would have been a spectacle for the ages! 🌪️🤼‍♂️

Who do YOU think would have come out on top? Drop your thoughts in the comments! 👇💬

#MMA #DreamMatch #FergusonVsKhabib #Legends #WhatIf 

=== FACEBOOK ===
🌟 **Remembering a Dream Match: Tony Ferguson Vs. Khabib Nurmagomedov** 🌟

Hey everyone! 😊 Let’s take a trip down memory lane. Imagine if Tony Ferguson and Khabib Nurmagomedov had faced off in the octagon. Can you believe that this iconic clash